# 13.1: Putting the knobs together

Implementations apply these transformations in a fixed order, usually: penalties on the raw logits, then temperature, then top-k, then top-p, then sampling. The order matters. Applying temperature before top-p means a high temperature enlarges the nucleus; applying it after does not. Here is a compact NumPy implementation:


In [ ]:
import numpy as np

def softmax(z):
    z = z - z.max()                      # subtract max for numerical stability
    e = np.exp(z)
    return e / e.sum()

def sample_next(logits, temperature=1.0, top_k=None, top_p=None, rng=np.random.default_rng()):
    z = np.asarray(logits, dtype=np.float64)
    if temperature == 0:                 # convention: temperature 0 means greedy
        return int(np.argmax(z))
    z = z / temperature
    if top_k is not None:                # keep the k largest logits
        kth = np.sort(z)[-top_k]
        z = np.where(z < kth, -np.inf, z)
    probs = softmax(z)
    if top_p is not None:                # keep the smallest set with mass >= top_p
        order = np.argsort(-probs)
        cum = np.cumsum(probs[order])
        cutoff = np.searchsorted(cum, top_p) + 1
        keep = order[:cutoff]
        mask = np.zeros_like(probs, dtype=bool)
        mask[keep] = True
        probs = np.where(mask, probs, 0.0)
        probs = probs / probs.sum()
    return int(rng.choice(len(probs), p=probs))

logits = np.array([4.0, 3.5, 2.0, 0.5, -1.0])
print(sample_next(logits, temperature=0))                  # always 0
print(sample_next(logits, temperature=0.7, top_p=0.9))     # 0 or 1, occasionally 2
